# 08 — Tuned XGBoost

**Goal:** find better XGBoost hyperparameters with randomized search (maximize CV ROC-AUC).

## Why this experiment?
Default settings are rarely optimal. Tuning depth, learning rate, subsample, etc. can unlock more performance on the same features.

## Approach
1. Use the full engineered feature set (pricing + time + geo).
2. Run RandomizedSearchCV (30 trials, 4-fold stratified CV) on training data only.
3. Refit the best params and score once on the held-out test set.

## What changed?
- Model: untuned XGB → **tuned XGB**
- Features: full FE (not raw)
- Search never sees the test set

## Features used in this notebook
- Full pricing + time + geo engineered set (same strong FE as later winners)
- **How selected:** lock the best FE from notebooks 03–06; only tune the model
- **Why:** ask “does better hyperparameters beat better features alone?”


### Setup
Shared data load and fixed split.


In [ ]:
import os
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "shared" / "protocol.py").exists():
        EXPERIMENT_ROOT = candidate
        break
    if (candidate / "hyperack_exp" / "shared" / "protocol.py").exists():
        EXPERIMENT_ROOT = candidate / "hyperack_exp"
        break
else:
    raise RuntimeError("Run this notebook from the HyperAck project directory.")
os.chdir(EXPERIMENT_ROOT)
sys.path.insert(0, str(EXPERIMENT_ROOT))

from shared.protocol import (
    add_geo_features,
    add_pricing_features,
    add_time_features,
    base_features,
    evaluate,
    load_clean_df,
    make_xy,
    save_result,
    split_frame,
    actual_vs_predicted_report,
)

RANDOM_STATE = 42
train_df, test_df = split_frame(load_clean_df())


### Features and CV
Full FE + stratified folds for tuning.


In [ ]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
X_train, y_train = make_xy(train_df, pricing=True, time_features=True, geo=True)
X_test, y_test = make_xy(test_df, pricing=True, time_features=True, geo=True)
cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=RANDOM_STATE)


### Features selected and why

Feature set is fixed to the **full engineered matrix**:

- Base raw columns  
- Pricing ratios / deltas / logs  
- Time + cyclical encodings  
- Haversine distance + bearing  

No further manual pruning here. Selection already happened by choosing the FE families that worked in 03–06.

Next cell lists every selected column.


In [ ]:
feature_why = {
    "deliverey_category_id": "Delivery type — some categories get accepted more often",
    "weekday": "Day of week — weekday vs weekend courier behavior",
    "time_bucket": "Coarse time-of-day bucket from the raw data",
    "total_distance": "Trip length — longer trips can be harder to accept",
    "sum_product": "Order size / number of products",
    "source_latitude": "Pickup latitude — area effects",
    "source_longitude": "Pickup longitude — area effects",
    "destination_latitude": "Drop-off latitude — area effects",
    "destination_longitude": "Drop-off longitude — area effects",
    "first_customer_fare": "First offered customer price (usually known early)",
    "final_customer_fare": "Final customer price — strong but may be post-decision",
    "final_biker_fare": "Final courier pay — strong but may be post-decision",
    "geo_cluster": "Train-only KMeans region of the trip (pickup+drop-off)",
    "log_distance": "Log distance — softens very long trips",
    "first_fare_per_km": "First fare ÷ distance — pay vs effort",
    "final_customer_fare_per_km": "Final customer fare ÷ distance",
    "customer_fare_delta": "Final − first customer fare (price change)",
    "customer_fare_change_pct": "Relative fare change vs first offer",
    "biker_customer_gap": "Biker fare − customer fare (split / margin)",
    "biker_fare_per_km": "Courier pay per km",
    "log_final_customer_fare": "Log of final customer fare",
    "log_final_biker_fare": "Log of final biker fare",
    "hour": "Exact hour of order creation",
    "is_rush_hour": "Lunch/evening peak flag",
    "is_weekend": "Weekend flag",
    "hour_sin": "Cyclical hour (sin) so 23 is near 0",
    "hour_cos": "Cyclical hour (cos)",
    "weekday_sin": "Cyclical weekday (sin)",
    "weekday_cos": "Cyclical weekday (cos)",
    "day_of_month": "Calendar day — mild monthly pattern",
    "haversine_km": "Great-circle route distance in km",
    "latitude_delta": "North/south trip span",
    "longitude_delta": "East/west trip span",
    "geo_bearing_sin": "Trip direction (sin of bearing)",
    "geo_bearing_cos": "Trip direction (cos of bearing)",
    "distance_x_first_fare": "Interaction: long trip × price",
    "category_x_hour": "Interaction: category × hour",
    "total_distance_qbin": "Train-fitted distance quantile bin",
    "first_customer_fare_qbin": "Train-fitted first-fare quantile bin"
}

cols = list(X_train.columns)
rows = []
for c in cols:
    rows.append({
        "feature": c,
        "why_selected": feature_why.get(c, "Part of this experiment's engineered feature set"),
    })
feature_table = pd.DataFrame(rows)
print(f"Total features selected: {len(cols)}")
print("Columns:")
print(", ".join(cols))
feature_table


### Hyperparameter search
30 random trials maximizing ROC-AUC.


In [ ]:
from scipy.stats import randint, uniform
from xgboost import XGBClassifier

search = RandomizedSearchCV(
    Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", XGBClassifier(eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1))]),
    param_distributions={
        "model__n_estimators": randint(300, 1200),
        "model__max_depth": randint(3, 10),
        "model__learning_rate": uniform(0.015, 0.135),
        "model__subsample": uniform(0.65, 0.35),
        "model__colsample_bytree": uniform(0.65, 0.35),
        "model__min_child_weight": randint(1, 12),
        "model__reg_lambda": uniform(0.1, 8.0),
    },
    n_iter=30, scoring="roc_auc", cv=cv, random_state=RANDOM_STATE, n_jobs=-1, refit=True,
)
model = search


### Evaluate best model
One held-out test evaluation.


In [ ]:
metrics = evaluate(model, X_train, y_train, X_test, y_test)
search.best_params_, metrics


### Actual vs predicted (test set)

After training, we score the **held-out test set** and compare:

1. **Actual** labels (`hyper_ack`) vs **predicted** labels  
2. Confusion matrix (rows = actual, columns = predicted)  
3. Per-class precision / recall / F1  
4. A sample of correct and incorrect rows with predicted probability  

This is only test-set performance — not training rows.


In [ ]:
from IPython.display import display
from shared.protocol import actual_vs_predicted_report

avp = actual_vs_predicted_report(
    metrics["y_true"],
    metrics["y_pred"],
    metrics["y_prob"],
    sample_size=25,
)
print("1) Actual vs predicted class counts")
display(avp["class_counts"])
print("2) Confusion matrix")
display(avp["confusion_matrix"])
print("3) Outcome breakdown")
display(avp["outcomes"])
print("4) Per-class metrics")
display(avp["per_class_metrics"])
print("5) Sample of actual vs predicted rows")
display(avp["prediction_sample"])


### Save result
Write experiment `08`.


In [ ]:
result_path = save_result(
    "08",
    "xgboost_tuned",
    "30-trial randomized-search XGBoost on full engineered features",
    metrics,
    best_model="RandomizedSearchCV(XGBClassifier)",
    notes="Validation tuning uses train folds only.",
    feature_count=X_train.shape[1],
)
pd.Series(metrics).drop("confusion_matrix").sort_index(), result_path


## What to look at
- Best params + held-out ROC-AUC
- Compare to untuned trees (02) and tuned LightGBM (09)
